# Generating Bay Area patient sample points

This notebook documents the methodology used to generate the patient location dataset consumed by the Julia simulation code (`CA_simulations.jl`, `decision_tree_build.jl`). The canonical 5,000-point output for the paper is checked in as **`../sampled_points/CA_points.csv`** — this notebook reproduces the methodology that generated it.

**Method.** Patient locations are sampled across the San Francisco Bay Area, weighted by 2019 population density (Meta/CIESIN high-resolution data via the Humanitarian Data Exchange). Sampled points are then constrained to land using a polygon shapefile.

## Data sources (download separately)

These files are NOT committed to the repo because they are large. Download them into `data/` before running:

- **Population density (.vrt + companion .tif/.vrt tiles):** https://data.humdata.org/dataset/united-states-high-resolution-population-density-maps-demographic-estimates
- **Bay Area ZIP code shapefile:** https://geodata.lib.berkeley.edu/catalog/ark28722-s7888q

See `README.md` in this folder for full setup instructions.

## Known limitation

The canonical `CA_points.csv` was generated using a Natural Earth land polygon for the final land filter, not the Berkeley ZIP shapefile shown here. This notebook documents the sampling and ZIP-based filter as a faithful methodology trace; if you regenerate, your output will not be schema-identical to `CA_points.csv`. See `README.md` for the full discussion.

## Setup and parameters

All tunable knobs live in this cell. Change them here, not buried in the code below.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import Point, box
import matplotlib.pyplot as plt

# --- Paths (relative to this notebook's location: repo/data_prep/) ---
DATA_DIR = Path("data")
VRT_FILE = DATA_DIR / "population_usa_2019-07-01.vrt"
BAY_AREA_SHP = DATA_DIR / "bayarea_zipcodes.shp"

OUT_DIR = Path("../sampled_points")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_POINTS_CSV = OUT_DIR / "CA_points_raw.csv"        # before land filter
FILTERED_POINTS_CSV = OUT_DIR / "CA_points_regenerated.csv"  # after land filter — does NOT overwrite canonical CA_points.csv

# --- Sampling parameters ---
NUM_SAMPLES = 5000      # total weighted samples to draw
RANDOM_SEED = 42        # for reproducibility
DROP_DUPLICATES = True  # set False to match the original (which left ~1% duplicates)

# --- Grid parameters (for downstream heatmap visualization, optional) ---
GRID_NX, GRID_NY = 50, 50
GRID_SAMPLES_PER_CELL = 100

# Seed both Python's `random` and NumPy's RNG.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Sampling {NUM_SAMPLES} points (seed={RANDOM_SEED}, drop_duplicates={DROP_DUPLICATES})")
print(f"Input population raster: {VRT_FILE}")
print(f"Input ZIP shapefile:     {BAY_AREA_SHP}")
print(f"Output (raw):            {RAW_POINTS_CSV}")
print(f"Output (filtered):       {FILTERED_POINTS_CSV}")

## Step 1 — Load population density data, masked to the Bay Area

Open the population raster and clip it to the Bay Area polygon. Returns a 2D array of population densities (NaN outside the polygon and on no-data cells), and the geo-transform needed to convert array indices back to lat/lon.

In [ ]:
def get_population_data(vrt_file_path, shape_file):
    bay_area_shape = gpd.read_file(shape_file)
    with rasterio.open(vrt_file_path) as src:
        if str(bay_area_shape.crs) != src.crs.to_string():
            bay_area_shape = bay_area_shape.to_crs(src.crs)
        out_image, out_transform = mask(src, bay_area_shape.geometry, crop=True)
        out_meta = src.meta.copy()
    population_data = out_image[0]
    no_data = out_meta["nodata"]
    population_data = np.where(population_data == no_data, np.nan, population_data)
    return population_data, out_transform

population_data, out_transform = get_population_data(VRT_FILE, BAY_AREA_SHP)
print(f"Population raster shape: {population_data.shape}")
print(f"Non-NaN cells: {np.sum(~np.isnan(population_data)):,}")

## Step 2 — Weighted random sample of points

Sample `NUM_SAMPLES` cells from the raster with probability proportional to population density, then convert array indices back to (longitude, latitude).

In [ ]:
def sample_points(population_data, out_transform, num_samples, rng):
    flat_population = population_data.flatten()
    indices = np.arange(flat_population.size)
    # NaN > 0 evaluates to False, so this also filters out no-data cells.
    valid_indices = indices[flat_population > 0]
    weights = flat_population[valid_indices]
    weights /= weights.sum()
    sampled_indices = rng.choice(valid_indices, size=num_samples, p=weights)
    sampled_rows, sampled_cols = np.unravel_index(sampled_indices, population_data.shape)
    return [rasterio.transform.xy(out_transform, r, c) for r, c in zip(sampled_rows, sampled_cols)]

rng = np.random.default_rng(RANDOM_SEED)
sampled_points = sample_points(population_data, out_transform, NUM_SAMPLES, rng)
print(f"Sampled {len(sampled_points)} points")
print("First 3:", sampled_points[:3])

## Step 3 — Inspect duplicates, optionally drop them

Because the population raster has discrete cells, two samples can collapse to identical lat/lon. The original paper data kept these (~1% rate). Set `DROP_DUPLICATES = False` in the parameters cell to reproduce that behavior; the default below drops them and prints the count.

In [ ]:
points_df = pd.DataFrame(sampled_points, columns=["longitude", "latitude"])

dup_mask = points_df.duplicated(subset=["longitude", "latitude"], keep=False)
print(f"Duplicate rows (including all repeats): {dup_mask.sum()}")

if DROP_DUPLICATES:
    before = len(points_df)
    points_df = points_df.drop_duplicates(subset=["longitude", "latitude"]).reset_index(drop=True)
    print(f"Dropped {before - len(points_df)} duplicate rows; {len(points_df)} unique points remain.")

points_df.to_csv(RAW_POINTS_CSV, index=False)
print(f"Wrote raw points (pre-land-filter) to {RAW_POINTS_CSV}")

## Step 4 — Filter to land via spatial join

Drop any sampled points that fall outside the Bay Area land polygon (e.g., on water — even though the population mask should already exclude these, edge cells can land in the bay).

In [ ]:
geometry = [Point(xy) for xy in zip(points_df.longitude, points_df.latitude)]
points_gdf = gpd.GeoDataFrame(points_df.copy(), geometry=geometry)

land_gdf = gpd.read_file(BAY_AREA_SHP)
if points_gdf.crs is None:
    points_gdf.set_crs(land_gdf.crs, inplace=True)
elif points_gdf.crs != land_gdf.crs:
    points_gdf = points_gdf.to_crs(land_gdf.crs)

# `op=` was deprecated in geopandas 0.10 and removed in 0.14; use `predicate=`.
points_on_land = gpd.sjoin(points_gdf, land_gdf, how="inner", predicate="intersects")
points_on_land = points_on_land.drop(columns="index_right")

print(f"Before land filter: {len(points_gdf)}")
print(f"After land filter:  {len(points_on_land)}")

points_on_land.to_csv(FILTERED_POINTS_CSV, index=False)
print(f"Wrote land-filtered points to {FILTERED_POINTS_CSV}")
print("NOTE: This file is named `CA_points_regenerated.csv` so it does NOT overwrite the canonical `CA_points.csv`.")

## Step 5 — Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
land_gdf.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.3)
points_on_land.plot(ax=ax, marker="o", color="red", markersize=2, alpha=0.6)
ax.set_title(f"Bay Area patient sample points (n={len(points_on_land)})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## (Optional) Step 6 — Uniform-per-cell grid sampling

The original notebook also produced a 50×50 grid of cells with 100 uniformly-sampled points per cell. This was used to visualize the optimal-vs-status-quo policy difference as a heatmap in `CA_grid_maker.jl`. **The simulation outputs in `sampled_points/grid_plot_csvs_CA/` are produced by the Julia simulation runs**, not by this step — so you only need to run this section if you want to verify the input-point methodology.

Skip this cell if you only need the canonical 5,000-point dataset.

In [ ]:
def create_grid(bounding_geom, num_x, num_y, crs):
    minx, miny, maxx, maxy = bounding_geom.bounds
    dx = (maxx - minx) / num_x
    dy = (maxy - miny) / num_y
    cells = [
        box(minx + x * dx, miny + y * dy, minx + (x + 1) * dx, miny + (y + 1) * dy)
        for x in range(num_x)
        for y in range(num_y)
    ]
    return gpd.GeoDataFrame(geometry=cells, crs=crs)

def sample_points_within_grid(grid, num_samples_per_cell, rng):
    pts = []
    for cell in grid.geometry:
        minx, miny, maxx, maxy = cell.bounds
        for _ in range(num_samples_per_cell):
            pts.append((rng.uniform(minx, maxx), rng.uniform(miny, maxy)))
    return pts

bay_area_shape = gpd.read_file(BAY_AREA_SHP)
# `unary_union` -> `union_all()` (shapely 2.x).
grid = create_grid(bay_area_shape.union_all(), GRID_NX, GRID_NY, bay_area_shape.crs)

grid_rng = random.Random(RANDOM_SEED)
grid_points = sample_points_within_grid(grid, GRID_SAMPLES_PER_CELL, grid_rng)
print(f"Generated {len(grid_points)} grid-sampled points ({GRID_NX}x{GRID_NY} cells x {GRID_SAMPLES_PER_CELL}/cell).")

grid_points_gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(*zip(*grid_points)),
    crs=bay_area_shape.crs,
)
grid_points_on_land = gpd.sjoin(grid_points_gdf, bay_area_shape, how="inner", predicate="intersects")
grid_points_on_land = grid_points_on_land.drop(columns="index_right")
print(f"After land filter: {len(grid_points_on_land)}")

fig, ax = plt.subplots(figsize=(12, 12))
bay_area_shape.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.3)
grid.plot(ax=ax, facecolor="none", edgecolor="blue", linewidth=0.3)
grid_points_on_land.plot(ax=ax, markersize=2, color="red", alpha=0.4)
ax.set_title(f"{GRID_NX}x{GRID_NY} grid with {GRID_SAMPLES_PER_CELL} samples/cell (n on land={len(grid_points_on_land)})")
plt.show()